# Affine vertex algebras

This notebook shows the universal affine vertex (super)algebras $V^k(\mathfrak g)$: their OPEs and the normalisation of the level, the four ways to give the Lie (super)algebra, the same algebra built by hand from structure constants, the Sugawara conformal vector, the grading by the root lattice, and the simple quotient $L_k(\mathfrak{sl}_2)$ at levels 1 and 2.

**Prerequisites:** `01_getting_started.ipynb`, `02_lie_superalgebras.ipynb`.

**Contents**

1. The OPEs, the invariant form and the level
2. Four ways to give the Lie algebra
3. Building it by hand
4. The Sugawara vector and the critical level
5. The root lattice grading
6. Singular vectors and the simple quotient at integrable level
7. Not there yet

## 1. The OPEs, the invariant form and the level

For a Lie (super)algebra $\mathfrak g$ with an invariant supersymmetric form $(\cdot|\cdot)$, the universal affine vertex algebra $V^k(\mathfrak g)$ is strongly generated by currents $J^a$, one for each basis vector $a$ of $\mathfrak g$, with

$$J^a(z)J^b(w) \sim \frac{k\,(a|b)}{(z-w)^2} + \frac{J^{[a,b]}(w)}{z-w}.$$

For a simple Lie algebra the form is the Killing form normalised so that the highest root $\theta$ has $(\theta|\theta) = 2$. `va.Affine(R, g, k)` takes the coefficient ring, the Lie algebra and the level. Given a Cartan type, the basis is Sage's Chevalley basis: `alpha[1]` is $e_{\alpha_1}$, `alphacheck[1]` is $h_1$, `-alpha[1]` is $f_{\alpha_1}$. The level here is a parameter $k$ in $\mathbb Q(k)$.

In [1]:
import vertex_algebras as va
from vertex_algebras import *

K.<k> = Frac(QQ['k'])
V = va.Affine(K, ['A', 1], k)
e, h, f = V.gens()
V.variable_names(), e.ope(f), h.ope(e), h.ope(h)

(('alpha[1]', 'alphacheck[1]', '-alpha[1]'),
 [alphacheck[1], k],
 [2*alpha[1]],
 [0, 2*k])

So $e(z)f(w) \sim k/(z-w)^2 + h(w)/(z-w)$, $h(z)e(w) \sim 2e(w)/(z-w)$ and $h(z)h(w) \sim 2k/(z-w)^2$. `invariant_form_matrix()` is the Gram matrix of $k(\cdot|\cdot)$ in the order of the generators, the second-order poles. `killing_form_matrix()` is the Killing form $\operatorname{tr}(\operatorname{ad}a \operatorname{ad}b)$, which is $2h^\vee = 4$ times the normalised form for $\mathfrak{sl}_2$.

In [2]:
V.invariant_form_matrix(), V.killing_form_matrix()

(
[  0   0   k]  [0 0 4]
[  0 2*k   0]  [0 8 0]
[  k   0   0], [4 0 0]
)

For $\mathfrak{sl}_3$ the generators are the six root vectors and two Cartan elements; the form pairs $e_\alpha$ with $e_{-\alpha}$, and on the Cartan subalgebra it is $k$ times the Cartan matrix. The level can also be a number, here $k = 5/2$ over $\mathbb Q$:

In [3]:
V3 = va.Affine(K, ['A', 2], k)
print(V3.variable_names())
print(V3.invariant_form_matrix())
W = va.Affine(QQ, ['A', 2], 5/2)
W.gen('alpha[1]').ope(W.gen('-alpha[1]'))

('alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]')
[  0   0   0   0   0   k   0   0]
[  0   0   0   0   0   0   k   0]
[  0   0   0   0   0   0   0   k]
[  0   0   0 2*k  -k   0   0   0]
[  0   0   0  -k 2*k   0   0   0]
[  k   0   0   0   0   0   0   0]
[  0   k   0   0   0   0   0   0]
[  0   0   k   0   0   0   0   0]


[alphacheck[1], 5/2]

For a Lie superalgebra (see `02_lie_superalgebras.ipynb`, and [Kac1977]) the currents of odd basis vectors are odd, and the form is normalised as in the tables of [KW2004]. In $\mathfrak{sl}(2|1)$ the simple root $\alpha_2$ is odd and isotropic: $x = $ `E[alpha[2]]` and $y = $ `E[-alpha[2]]` are odd, $x(z)x(w) \sim 0$, and the second-order pole changes sign when the two odd currents are exchanged, as supersymmetry of the form requires.

In [4]:
S = LieSuperalgebra(QQ, 'sl(2|1)')
VS = va.Affine(K, S, k)
x, y = VS.gen('E[alpha[2]]'), VS.gen('E[-alpha[2]]')
x.parity(), x.ope(y), y.ope(x), x.ope(x)

(1, [h2, k], [h2, -k], [])

In $\mathfrak{osp}(1|2)$ (`spo(2|1)`) the odd simple root is not isotropic: $[x, x] = e_{2\alpha}$, so $x(z)x(w) \sim e_{2\alpha}(w)/(z-w)$.

In [5]:
O = va.Affine(K, LieSuperalgebra(QQ, 'spo(2|1)'), k)
xo = O.gen('E[alpha[1]]')
O.variable_names(), xo.ope(xo), xo.ope(O.gen('E[-alpha[1]]'))

(('E[alpha[1]]', 'E[2*alpha[1]]', 'h1', 'E[-alpha[1]]', 'E[-2*alpha[1]]'),
 [E[2*alpha[1]]],
 [h1, 4*k])

## 2. Four ways to give the Lie algebra

`va.Affine(R, g, k, form=None, ...)` accepts
- a Cartan type or Cartan matrix of a simple Lie algebra (section 1);
- a `LieSuperalgebra`, also one without odd roots;
- a Lie algebra of Sage with a finite basis, together with the invariant form `form` (a function of two elements, or a Gram matrix);
- structure constants `{(a, b): {c: coefficient}}`, with `parities={a: 1}` for the odd basis vectors and `form` a dict, a matrix or a function.

For the last two the level multiplies `form`. The constructor checks the (super) Jacobi identity and the invariance of the form, which are exactly the Jacobi identities of the vertex algebra. Here is $\mathfrak{sl}_2$ in all four ways. For Sage's Lie algebra the normalised form is the Killing form divided by $2h^\vee = 4$.

In [6]:
g = LieAlgebra(QQ, cartan_type=['A', 1])
sl2 = {('h', 'e'): {'e': 2}, ('h', 'f'): {'f': -2}, ('e', 'f'): {'h': 1}}
V_cartan = va.Affine(K, ['A', 1], k)
V_super = va.Affine(K, LieSuperalgebra(QQ, CartanMatrix(['A', 1])), k)
V_sage = va.Affine(K, g, k, form=lambda a, b: a.killing_form(b) / 4)
V_struct = va.Affine(K, sl2, k, form={('e', 'f'): 1, ('h', 'h'): 2}, names=['e', 'h', 'f'])
for A in (V_super, V_sage, V_struct):
    print(A.variable_names(), A.gen(0).ope(A.gen(2)), A.gen(1).ope(A.gen(0)))

('E[alpha[1]]', 'h1', 'E[-alpha[1]]') [h1, k] [2*E[alpha[1]]]
('alpha[1]', 'alphacheck[1]', '-alpha[1]') [alphacheck[1], k] [2*alpha[1]]
('e', 'h', 'f') [h, k] [2*e]


They are four objects (the data differ, at least in the names), but the same vertex algebra: the generators are in the order $e, h, f$ in each, and `V.hom(images)` checks that sending generators to generators respects all OPEs, and `hom` raises an error if they do not. So each call below succeeds and gives an isomorphism.

In [7]:
[V_cartan.hom(A.gens()).codomain() is A for A in (V_super, V_sage, V_struct)]

[True, True, True]

A Lie superalgebra given by structure constants: `structure_coefficients()` of $\mathfrak{sl}(2|1)$ is in the format the constructor takes, and `invariant_form_matrix()` gives the form in the basis order. The parities are read off the basis. The result is the algebra `VS` of section 1, up to the grading, which only the `LieSuperalgebra` input provides.

In [8]:
names = [str(b) for b in S.basis().keys()]
odd = {str(b): 1 for b in S.basis().keys() if S.basis()[b].is_even_odd()}
VS_struct = va.Affine(K, S.structure_coefficients(), k, form=S.invariant_form_matrix(),
                      parities=odd, names=names)
print(VS_struct)
VS.hom(VS_struct.gens()).codomain() is VS_struct, VS_struct.horizontal_grading(), VS.horizontal_grading()

Universal affine vertex superalgebra of a Lie superalgebra of dimension 8 over Fraction Field of Univariate Polynomial Ring in k over Rational Field


(True,
 None,
 {'E[alpha[1]]': (1, 0),
  'E[alpha[2]]': (0, 1),
  'E[alpha[1] + alpha[2]]': (1, 1),
  'h1': (0, 0),
  'h2': (0, 0),
  'E[-alpha[1]]': (-1, 0),
  'E[-alpha[2]]': (0, -1),
  'E[-alpha[1] - alpha[2]]': (-1, -1)})

Wrong data are refused. Here the bracket $[e, f]$ is doubled, so the form is no longer invariant:

In [9]:
bad = dict(sl2); bad[('e', 'f')] = {'h': 2}
va.Affine(K, bad, k, form={('e', 'f'): 1, ('h', 'h'): 2})

ValueError: the form is not invariant: ([e, f], h) = 4*k but (e, [f, h]) = 2*k

## 3. Building it by hand

The OPEs of section 1 can be written out directly as a `UniversalVertexAlgebra` (see `01_getting_started.ipynb`): one generator for each Chevalley basis vector of Sage's $\mathfrak{sl}_3$, first-order pole the bracket, second-order pole $k$ times the normalised Killing form. The normalisation factor is $1/(e_\theta|e_{-\theta})_{\text{Killing}}$, which is $1/(2h^\vee) = 1/6$. The OPE coefficients are elements of `NormallyOrderedExpressions`, and each pair of generators is given once.

In [10]:
g3 = LieAlgebra(QQ, cartan_type=['A', 2])
B = g3.basis(); keys = list(B.keys()); labels = [str(a) for a in keys]
theta = g3.cartan_type().root_system().root_lattice().highest_root()
s = 1 / B[theta].killing_form(B[-theta])

X = NormallyOrderedExpressions(K, labels)
def J(v):      # the current of a Lie algebra element
    return X.sum(c * X.gen(str(a)) for a, c in v.monomial_coefficients().items())

opes = {(labels[i], labels[j]): [J(B[a].bracket(B[b])), k * s * B[a].killing_form(B[b])]
        for i, a in enumerate(keys) for j, b in enumerate(keys) if i <= j}
Vhand = UniversalVertexAlgebra(K, X, opes, degrees={a: 1 for a in labels})
s, Vhand

(1/6,
 Universal vertex algebra on generators ['alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field)

The generators come in the same order as in `V3 = va.Affine(K, ['A', 2], k)`, so the identity on generators is a morphism, and the weight spaces have the same dimensions:

In [11]:
(V3.hom(Vhand.gens()).codomain() is Vhand,
 [Vhand.weight_space(n).dimension() for n in range(4)],
 [V3.weight_space(n).dimension() for n in range(4)])

(True, [1, 8, 44, 192], [1, 8, 44, 192])

## 4. The Sugawara vector and the critical level

For a simple Lie algebra and $k \ne -h^\vee$, the Sugawara vector [Kac1998]

$$T = \frac{1}{2(k + h^\vee)} \sum_a {:}J^a J_a{:},$$

with $J_a$ the currents of the dual basis for $(\cdot|\cdot)$, is a conformal vector of central charge $c = k\dim\mathfrak g/(k + h^\vee)$ for which all currents have conformal weight 1. It is a choice, made with `conformal_vector='sugawara'`. For $\mathfrak{sl}_2$, with generators named $E, H, F$ by `names=`:

In [12]:
Vs = va.Affine(K, ['A', 1], k, names=['E', 'H', 'F'], conformal_vector='sugawara')
E, H, F = Vs.gens()
T = Vs.conformal_vector()
T, T == (E.no(F) + F.no(E) + H.no(H) / 2) / (2 * (k + 2))

((1/(k + 2))*:E F: + (1/4/(k + 2))*:H H: + (-1/2/(k + 2))*dH, True)

The central charge is $3k/(k+2)$; $T(z)E(w) \sim E(w)/(z-w)^2 + \partial E(w)/(z-w)$ says that $E$ is primary of weight 1. For $\mathfrak{sl}_3$, $c = 8k/(k+3)$.

In [13]:
(Vs.central_charge(), Vs.degrees(), T.ope(E),
 va.Affine(K, ['A', 2], k, conformal_vector='sugawara').central_charge())

(3*k/(k + 2), {'E': 1, 'H': 1, 'F': 1}, [dE, E], 8*k/(k + 3))

For a Lie superalgebra the library uses $T = \tfrac12\sum_{a,b}\Omega_{ab}{:}J_aJ_b{:}$ with $\Omega = (B + \kappa/2)^{-1}$, $B$ the form times the level and $\kappa$ the super Killing form; for simple $\mathfrak g$ this is the formula above. The central charge is $c = k\operatorname{sdim}\mathfrak g/(k + h^\vee)$ [KRW2003]. For $\mathfrak{sl}(2|1)$, $\operatorname{sdim} = 0$ so $c = 0$; for $\mathfrak{osp}(1|2)$, $\operatorname{sdim} = 1$ and $h^\vee = 3/2$:

In [14]:
[(G, va.Affine(K, G, k, conformal_vector='sugawara').central_charge(), G.superdimension(), G.dual_coxeter_number())
 for G in (LieSuperalgebra(QQ, 'sl(2|1)'), LieSuperalgebra(QQ, 'spo(2|1)'))]

[(Lie superalgebra sl(2|1) over Rational Field, 0, 0, 1),
 (Lie superalgebra spo(2|1) over Rational Field, k/(k + 3/2), 1, 3/2)]

A rational level works the same way: $k = 1/2$ gives $c = (3/2)/(5/2) = 3/5$.

In [15]:
va.Affine(QQ, ['A', 1], 1/2, conformal_vector='sugawara').central_charge()

3/5

**The critical level.** At $k = -h^\vee$ the factor $1/(k + h^\vee)$ does not exist. The library refuses the request with a `ValueError` whenever $B + \kappa/2$ is not invertible over the coefficient ring: at the critical level, and also over a polynomial ring such as $\mathbb Q[k]$, where $k + h^\vee$ has no inverse.

In [16]:
va.Affine(QQ, ['A', 1], -2, conformal_vector='sugawara')

ValueError: no Sugawara vector: B + κ/2 is not invertible over Rational Field (the critical level?)

At the critical level the unnormalised vector $S = {:}EF{:} + {:}FE{:} + \frac12{:}HH{:}$ has no singular part in its OPE with any current: it commutes with all of $V^{-2}(\mathfrak{sl}_2)$. At $k = 1$ instead $J(z)S(w) \sim 6J(w)/(z-w)^2$: there $S = 2(k + h^\vee)T = 6T$, and $J(z)T(w) \sim J(w)/(z-w)^2$ for every current $J$.

In [17]:
for lev in (-2, 1):
    A = va.Affine(QQ, ['A', 1], lev, names=['E', 'H', 'F'])
    a, b, c = A.gens()
    S2 = a.no(c) + c.no(a) + b.no(b) / 2
    print(lev, [x.ope(S2) for x in A.gens()])

-2 [[], [], []]
1 [[0, 6*E], [0, 6*H], [0, 6*F]]


## 5. The root lattice grading

Given a Cartan type or a `LieSuperalgebra`, the algebra is graded by the root lattice: the current of a root vector has the root as its grade, written in simple root coordinates, and Cartan currents have grade 0. Grades add under all products. `x.horizontal_grade()` is the grade of a homogeneous element, and `V.horizontal_grading_group()` the group of grades ($\mathbb Z^{\text{rank}}$).

In [18]:
E.horizontal_grade(), H.horizontal_grade(), F.horizontal_grade(), E.no(E).horizontal_grade(), Vs.horizontal_grading_group()

((1),
 (0),
 (-1),
 (2),
 Ambient free module of rank 1 over the principal ideal domain Integer Ring)

`V.weight_space(n, horizontal_grade=a)` is the part of conformal weight $n$ and grade $a$. For $\mathfrak{sl}_2$, the dimensions for grades $-n, \dots, n$ at weights $n = 0, \dots, 3$:

In [19]:
[[Vs.weight_space(n, horizontal_grade=(a,)).dimension() for a in range(-n, n + 1)] for n in range(4)]

[[1], [1, 1, 1], [1, 2, 3, 2, 1], [1, 2, 5, 6, 5, 2, 1]]

In [20]:
Vs.weight_space(2, horizontal_grade=(2,)).basis(), Vs.weight_space(2, horizontal_grade=(1,)).basis()

([:E E:], [:E H:, dE])

`V.character(N)` is $\sum_{n<N}\dim V_n\,q^n$, and with `horizontal_grade=` the character of one grade. As a vector space $V^k(\mathfrak g)$ is the symmetric algebra on the negative modes of the currents, so its character is $\prod_{n\ge1}(1-q^n)^{-\dim\mathfrak g}$, here $\prod_{n \ge 1}(1-q^n)^{-3}$:

In [21]:
ch = Vs.character(7)
P.<q> = PowerSeriesRing(QQ, default_prec=7)
ch, ch.laurent_part() == (1 / prod((1 - q^n)^3 for n in range(1, 7))).truncate(7), Vs.character(7, horizontal_grade=(0,))

(1 + 3*q + 9*q^2 + 22*q^3 + 51*q^4 + 108*q^5 + 221*q^6 + O(q^7),
 True,
 1 + q + 3*q^2 + 6*q^3 + 13*q^4 + 24*q^5 + 47*q^6 + O(q^7))

For $\mathfrak{sl}_3$ the grades have two coordinates. Here the generators are named $e_i, h_i, f_i$ (the default names such as `alpha[1] + alpha[2]` contain spaces and signs, so that $\partial$ of `alpha[1] + alpha[2]` prints as `dalpha[1] + alpha[2]`). The highest root $\theta = \alpha_1 + \alpha_2$ has grade $(1, 1)$; at weight 2 it carries four states, and grade $(0,0)$ at weight 2 has dimension 8 (three ${:}e_\alpha f_\alpha{:}$, three ${:}h_ih_j{:}$, two $\partial h_i$).

In [22]:
V3s = va.Affine(K, ['A', 2], k, names=['e2', 'e1', 'e12', 'h1', 'h2', 'f2', 'f1', 'f12'],
                conformal_vector='sugawara')
print([(x, x.horizontal_grade()) for x in V3s.gens()])
V3s.weight_space(2, horizontal_grade=(1, 1)).basis(), V3s.weight_space(2, horizontal_grade=(0, 0)).dimension()

[(e2, (0, 1)), (e1, (1, 0)), (e12, (1, 1)), (h1, (0, 0)), (h2, (0, 0)), (f2, (0, -1)), (f1, (-1, 0)), (f12, (-1, -1))]


([:e2 e1:, :e12 h1:, :e12 h2:, de12], 8)

## 6. Singular vectors and the simple quotient at integrable level

For a positive integer $k$, the simple quotient $L_k(\mathfrak g)$ of $V^k(\mathfrak g)$ is the quotient by the ideal generated by the singular vector $e_\theta(-1)^{k+1}\mathbf 1$ [FZ1992]. For $\mathfrak{sl}_2$ this is ${:}E^{k+1}{:}$.

`V.find_singular(n, horizontal_grade=a, zero_modes=[...])` computes a basis of the singular vectors of weight $n$ and grade $a$: the vectors killed by the positive modes of the generators and by the zero modes of the listed elements (here $E$, for highest weight vectors). At $k = 1$ there is one of weight 2, ${:}EE{:}$, and none at weight 1.

In [23]:
V1 = va.Affine(QQ, ['A', 1], 1, names=['E', 'H', 'F'], conformal_vector='sugawara')
E1, H1, F1 = V1.gens()
(V1.find_singular(2, horizontal_grade=(2,), zero_modes=[E1]),
 [V1.find_singular(1, horizontal_grade=(a,), zero_modes=[E1]) for a in (-1, 0, 1)])

([:E E:], [[], [], []])

`V.ideal([v])` is the ideal generated by $v$, and `V.quotient(I)` (also `V / I`) the quotient. A quotient has no conformal vector or grading unless asked for: `conformal_vector='ambient'` takes the image of the Sugawara vector, and `horizontal_grading='ambient'` the root lattice grading (see `06_subalgebras_ideals_quotients.ipynb`). In $L_1(\mathfrak{sl}_2)$, ${:}EE{:} = 0$:

In [24]:
I = V1.ideal([E1.no(E1)])
L1 = V1.quotient(I, horizontal_grading='ambient', conformal_vector='ambient')
Eq, Hq, Fq = L1.gens()
[I.weight_space(n).dimension() for n in range(5)], L1.central_charge(), Eq.no(Eq), Eq.ope(Fq)

([0, 0, 5, 15, 38], 1, 0, [H, 1])

An independent check: $L_1(\mathfrak{sl}_2)$ is the lattice vertex algebra $V_{A_1}$ of the lattice $\mathbb Z\alpha$, $(\alpha|\alpha) = 2$ [FK1980] (see `05_lattice_vertex_algebras.ipynb`), with $E \mapsto e^{\alpha}$, $H \mapsto J$ (the Heisenberg current) and $F \mapsto e^{-\alpha}$. Its conformal vector is $\frac14{:}JJ{:}$ and its character is $\sum_{m\in\mathbb Z} q^{m^2}/\prod_{n\ge1}(1-q^n)$. The characters agree, also grade by grade (lattice grade $m$ is root grade $m$), and `hom` confirms that the map above is a morphism $L_1(\mathfrak{sl}_2) \to V_{A_1}$.

In [25]:
LA = va.Lattice(QQ, matrix([[2]])); Jl = LA.J(0)
LA = LA.with_conformal_vector(Jl.no(Jl) / 4).with_horizontal_grading('lattice')
print(L1.character(9))
print(LA.character(9))
print(L1.character(9) == LA.character(9),
      all(L1.character(9, horizontal_grade=(m,)) == LA.character(9, horizontal_grade=m) for m in range(-3, 4)))
L1.hom([LA.e((1,)), Jl, LA.e((-1,))])

1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + O(q^9)
1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + O(q^9)
True True


Vertex algebra morphism:
  From: Quotient of Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field by the ideal generated by [:E E:]
  To:   Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
  Defn: E |--> e^(1)
        H |--> J0
        F |--> e^(-1)

Note that the lattice algebra's grading group is $\mathbb Z$, so its grades are integers, while the affine algebra's is $\mathbb Z^1$, with grades written `(m,)`.

At $k = 2$ the singular vector is ${:}EEE{:}$ (`V.no(E, E, E)`), of weight 3. The character of $L_2(\mathfrak{sl}_2)$ can be compared with the integer-weight part of the character $\prod_{n\ge1}(1+q^{n-1/2})^3$ of three free fermions, whose bilinears ${:}\psi_i\psi_j{:}$ generate a copy of $L_2(\mathfrak{sl}_2)$. In the series below $x = q^{1/2}$.

In [26]:
V2 = va.Affine(QQ, ['A', 1], 2, names=['E', 'H', 'F'], conformal_vector='sugawara')
E2 = V2.gen('E')
sing = V2.find_singular(3, horizontal_grade=(3,), zero_modes=[E2])
L2 = V2.quotient(sing, conformal_vector='ambient')
Px.<x> = PowerSeriesRing(QQ, default_prec=20)
fermions = prod((1 + x^(2*n - 1))^3 for n in range(1, 10))
sing, L2.central_charge(), L2.character(9), [fermions[2*n] for n in range(9)]

([:E E E:],
 3/2,
 1 + 3*q + 9*q^2 + 15*q^3 + 30*q^4 + 54*q^5 + 94*q^6 + 153*q^7 + 252*q^8 + O(q^9),
 [1, 3, 9, 15, 30, 54, 94, 153, 252])

Zhu algebras of these quotients, and with them the classification of their modules, are in `09_zhu_algebras.ipynb`. The research example `13_singular_vectors_in_affine_vertex_algebras.ipynb` searches for singular vectors of $V^k(D_4)$ at the non-integrable level $k = -2$.

## 7. Not there yet

- Sage's super Cartan types (such as `['A', [1, 0]]`) are refused with a `NotImplementedError`; give a `LieSuperalgebra` or structure constants instead.
- The Sugawara vector exists only when $B + \kappa/2$ is invertible over the coefficient ring: not at the critical level, and not over a polynomial ring in $k$ (use its fraction field).
- Only Cartan types and `LieSuperalgebra`s come with the root lattice grading; for a Sage Lie algebra or structure constants pass `horizontal_grading=` yourself.

## References

- [FK1980] I. B. Frenkel, V. G. Kac, *Basic representations of affine Lie algebras and dual resonance models*, Invent. Math. 62 (1980), 23–66, [doi:10.1007/BF01391662](https://doi.org/10.1007/BF01391662) — $L_1(\mathfrak{sl}_2)$ as the $A_1$ lattice vertex algebra (section 6).
- [FZ1992] I. B. Frenkel, Y. Zhu, *Vertex operator algebras associated to representations of affine and Virasoro algebras*, Duke Math. J. 66 (1992), 123–168, [doi:10.1215/S0012-7094-92-06604-X](https://doi.org/10.1215/S0012-7094-92-06604-X) — the simple quotients at positive integer level (section 6).
- [Kac1977] V. G. Kac, *Lie superalgebras*, Adv. Math. 26 (1977), 8–96, [doi:10.1016/0001-8708(77)90017-2](https://doi.org/10.1016/0001-8708(77)90017-2) — basic classical Lie superalgebras (section 1).
- [Kac1998] V. G. Kac, *Vertex algebras for beginners*, 2nd ed., Univ. Lecture Ser. 10, AMS (1998), [doi:10.1090/ulect/010](https://doi.org/10.1090/ulect/010) — affine vertex algebras and the Sugawara construction (section 4).
- [KRW2003] V. G. Kac, S.-S. Roan, M. Wakimoto, *Quantum reduction for affine superalgebras*, Comm. Math. Phys. 241 (2003), 307–342, [arXiv:math-ph/0302015](https://arxiv.org/abs/math-ph/0302015) — the Sugawara vector of affine vertex superalgebras and its central charge (section 4).
- [KW2004] V. G. Kac, M. Wakimoto, *Quantum reduction and representation theory of superconformal algebras*, Adv. Math. 185 (2004), 400–458; corrigendum, Adv. Math. 193 (2005), 453–455, [arXiv:math-ph/0304011](https://arxiv.org/abs/math-ph/0304011) — the normalisation of the invariant forms of Lie superalgebras (section 1).